In [1]:
import os

In [2]:
%pwd

'd:\\Text-Summerizer-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Text-Summerizer-project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name= config.tokenizer_name
        )
        
        return data_transformation_config

In [8]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

d:\Text-Summerizer-project\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [16]:
from transformers import AutoTokenizer
import os
from datasets import load_from_disk

class DataTransformation:
    def __init__(self, config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        # Input (dialogue)
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=512,
            truncation=True,
            padding="max_length"
        )

        # Target (summary) - modern way
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [17]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation_config = DataTransformation(config=data_transformation_config)
    data_transformation_config.convert()
except Exception as e:
    raise e

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 440, in format
    return self._format(record)
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 436, in _format
    return self._fmt % values
KeyError: 'modules'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1100, in emit
    msg = self.format(record)
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 943, in format
    return fmt.format(record)
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 681, in format
    s = self.formatMessage(record)
  File "C:\Users\mahen\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 650, in formatMessage
